# 13. Accuracy@10 (tasa de acierto)

Para cumplir con la métrica de *accuracy* solicitada, se calcula la **tasa de acierto (HitRate@10)**: el porcentaje de usuarios para los que al menos uno de los 10 productos recomendados aparece entre los productos nuevos con los que interactuaron en el período de prueba. Se compara con la línea base de popularidad. Es la forma de accuracy apropiada para un sistema de recomendación, ya que una clasificación binaria tradicional no se aplica a rankings.

In [1]:
import numpy as np
import pandas as pd

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
d = np.load("modelo/modelo_als.npz")
e = np.load("modelo/extras.npz")
U, V, vistos, popular = d["user_factors"], d["item_factors"], e["vistos"], e["popular"]
K = 10

train_items = train.groupby("user_id")["item_id"].apply(set)
test_items = test.groupby("user_id")["item_id"].apply(set)

def top_als(u):
    s = U[u] @ V.T
    s[vistos[u]] = -np.inf
    return list(np.argsort(-s)[:K])

def top_pop(u):
    v = train_items.get(u, set())
    return [i for i in popular if i not in v][:K]

def hit_rate(fn):
    aciertos, n = 0, 0
    for u, rel in test_items.items():
        rel = rel - train_items.get(u, set())
        if not rel:
            continue
        n += 1
        aciertos += int(any(i in rel for i in fn(u)))
    return aciertos / n, n

for nombre, fn in [("ALS", top_als), ("Popularidad", top_pop)]:
    tasa, n = hit_rate(fn)
    print(f"{nombre:<12} HitRate@10 (accuracy@10): {tasa:.4f}  (usuarios evaluados: {n})")

ALS          HitRate@10 (accuracy@10): 0.2537  (usuarios evaluados: 954)
Popularidad  HitRate@10 (accuracy@10): 0.2180  (usuarios evaluados: 954)


### Interpretación de la tasa de acierto

Sobre los **954 usuarios evaluados**, el modelo ALS obtuvo un **HitRate@10 de 0,2537** (242 usuarios con al menos un producto relevante entre sus 10 recomendaciones), frente a **0,2180** de la línea base de popularidad (208 usuarios). La diferencia es de 3,6 puntos porcentuales, equivalente a una ventaja relativa de aproximadamente un 16%, consistente con la observada en Precision@10, Recall@10, NDCG@10 y F1@10.

Esta métrica se utiliza como equivalente de *accuracy* en un sistema de recomendación, donde la clasificación binaria tradicional no aplica. Debe interpretarse con cautela: los datos son sintéticos, la evaluación usa una única partición temporal y no se realizó una prueba de significancia estadística, por lo que el resultado se considera una línea base (champion) contra la cual comparar los reentrenamientos semanales.